# A parser for Propositional Logic

This notebook implements a parser that converts propositional logic formulas into `Tuple` structures defined in `recursive-set.ts`.

In [ ]:
import { Tuple } from "recursive-set";

## Domain Types

In [ ]:
type Variable = string;
type Operator = '↔' | '→' | '∧' | '∨' | '⊕';
type UnaryOp  = '¬';
type ConstOp  = '⊤' | '⊥';


const PRECEDENCE: Record<string, number> = {
    '↔': 1,
    '→': 2,
    '⊕': 3,
    '∨': 4,
    '∧': 5,
    '¬': 6,
    '⊤': 7,
    '⊥': 7
};

## Structural Classes

In [ ]:
class Constant extends Tuple<[ConstOp]> {
    constructor(val: ConstOp) { super(val); }
    get value() { return this.get(0); } // Return Type inferred as ConstOp
}

class Negation<T extends Formula = Formula> extends Tuple<['¬', T]> {
    constructor(phi: T) { super('¬', phi); }
    
    get phi(): T { return this.get(1); }
}

class BinaryFormula<L extends Formula = Formula, R extends Formula = Formula> 
    extends Tuple<[Operator, L, R]> {
    
    constructor(op: Operator, left: L, right: R) {
        super(op, left, right);
    }

    get operator() { return this.get(0); } 
    get left(): L  { return this.get(1); } // Return Type ist jetzt L
    get right(): R { return this.get(2); } // Return Type ist jetzt R
}

type Formula = Variable | Constant | Negation | BinaryFormula;

## Tokenizer

We use the same tokenizer logic as the original parser.

In [ ]:
const LEX_TOKENIZER = /([ \t]+)|([A-Za-z][A-Za-z0-9<>,]*)|([⊤⊥∧∨¬→↔⊕()])/g;

function tokenize(s: string): string[] {
    return Array.from(s.matchAll(LEX_TOKENIZER))
        .map(([_, _ws, ident, op]) => ident || op)
        .filter((t): t is string => !!t);
}

function isPropVar(s: string): boolean {
    return /^[A-Za-z][A-Za-z0-9<>,]*$/.test(s);
}

In [ ]:
tokenize('p ↔ q ↔ r')

## The Parser

In [ ]:
class LogicParser {
    private tokens: string[];
    private operators: string[] = [];
    private argumentsList: Formula[] = [];
    private input: string;

    constructor(s: string) {
        this.input = s;
        // Reverse tokens to use pop() (O(1)) instead of shift() (O(N))
        this.tokens = tokenize(s).reverse();
    }

    parse(): Formula {
        while (this.tokens.length > 0) {
            const token = this.tokens.pop()!;
            
            if (isPropVar(token)) {
                this.argumentsList.push(token);
            } 
            else if (token === '⊤' || token === '⊥') {
                this.argumentsList.push(new Constant(token));
            }
            else if (token === '(') {
                this.operators.push(token);
            }
            else if (token === ')') {
                let top = this.peekOperator();
                while (top !== undefined && top !== '(') {
                    this.popAndEvaluate();
                    top = this.peekOperator();
                }
                this.operators.pop();
            }
            else {
                while (
                    this.operators.length > 0 && 
                    this.operators[this.operators.length - 1] !== '(' &&
                    this.shouldPop(this.operators[this.operators.length - 1], token)
                ) {
                    this.popAndEvaluate();
                }
                this.operators.push(token);
            }
        }
        
        while (this.operators.length > 0) {
            this.popAndEvaluate();
        }
        
        if (this.argumentsList.length !== 1) {
            throw new Error(`Parse Error: Invalid Formula "${this.input}". Stack: ${this.argumentsList}`);
        }
        
        return this.argumentsList.pop()!;
    }

    private peekOperator(): string | undefined {
        return this.operators[this.operators.length - 1];
    }

    private shouldPop(stackOp: string, currentOp: string): boolean {
        const p1 = PRECEDENCE[stackOp] || 0;
        const p2 = PRECEDENCE[currentOp] || 0;

        if (p1 > p2) return true;
        if (p1 < p2) return false;

        return stackOp !== '→' && stackOp !== '¬';
    }

    private popAndEvaluate(): void {
        const op = this.operators.pop();
        if (!op) return;

        switch (op) {
            case '¬': {
                const arg = this.argumentsList.pop();
                if (!arg) throw new Error("Syntax Error: '¬' expects an operand.");
                this.argumentsList.push(new Negation(arg));
                break;
            }

            case '∧':
            case '∨':
            case '→':
            case '↔':
            case '⊕': {
                const right = this.argumentsList.pop();
                const left = this.argumentsList.pop();
                
                if (!left || !right) throw new Error(`Syntax Error: '${op}' expects 2 operands.`);
                
                this.argumentsList.push(new BinaryFormula(op, left, right));
                break;
            }

            default:
                throw new Error(`Unknown operator: ${op}`);
        }
    }
}

In [ ]:
function testParser(s) : void {
    const p = new LogicParser(s);
    console.log(s);
    console.log(p.parse());
}

In [ ]:
testParser('¬⊥')

In [ ]:
testParser('¬p ↔ (p → ⊥)')

In [ ]:
testParser('¬⊥ ↔ ⊤')

In [ ]:
testParser('p ∧ q')

In [ ]:
testParser('p ∨ q ∧ r')

In [ ]:
testParser('p ∧ q ∨ r')

In [ ]:
testParser('p ∧ q → r ∨ s')

In [ ]:
testParser('p → q → r')

In [ ]:
testParser('p ∧ q ↔ q ∨ p')

In [ ]:
testParser('¬(p ∨ q) ↔ ¬p ∨ ¬q')

In [ ]:
testParser('¬(p ⊕ q) ↔ (p ↔ q)')

In [ ]:
testParser('a<1,2> ↔ b<2,1>')